# Plant Disease Classification

This notebook contains the code for a Plant disease classification project using CNN.

In [1]:

# Core TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras

# Import layers, models, optimizers, and losses from Keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization, 
    Input, LSTM, GRU, Embedding, RandomFlip, RandomRotation, RandomZoom, RandomCrop
)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import SparseCategoricalCrossentropy

# Callbacks for training
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Metrics
from tensorflow.keras.metrics import Accuracy, Precision, Recall

# Utilities
import numpy as np
import matplotlib.pyplot as plt

print("All required libraries are imported Successfully")

In [2]:
IMG_SIZE = 256
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 54


print("All required constants are imported Successfully")

In [3]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "data",
    shuffle=True,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)


print("Dataset is accessed Successfully")

NotFoundError: Could not find directory data

In [ ]:
# Class names
class_names = valid_ds.class_names
print(class_names)


print("CLasses for distinguishing are loaded.")

In [ ]:
# Define the resize and rescale pipeline
resize_rescale = Sequential([
    Resizing(IMG_SIZE, IMG_SIZE),  # Resize images to the target size
    Rescaling(1.0 / 255)           # Rescale pixel values to [0, 1]
])


print("Function for resizing and rescaling is created.")

In [ ]:
data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal_and_vertical"),
    RandomRotation(0.2),
    RandomZoom(0.2)
])

print("Function for data augmentation is created.")

In [ ]:
dataset = dataset.map(lambda x, y: (resize_rescale(x), y))

print("Dataset is maped to resize and rescale function.")

In [1]:
# Optimize dataset loading using cache and prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_size = int(0.8 * len(valid_ds))
val_size = int(0.1 * len(valid_ds))
test_size = len(valid_ds) - train_size - val_size


train_ds = dataset.take(train_size)
# Validation dataset
valid_ds = dataset.skip(train_size).take(val_size)
# Test dataset
test_ds = dataset.skip(train_size + val_size)

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
valid_ds = valid_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)


print("dataset loading of training,validation and test data is done by using cache and prefetch.")


NameError: name 'tf' is not defined

In [ ]:
# Model definition
model = model = Sequential([
    resize_rescale,
    data_augmentation,
    
    # Convolutional layers with BatchNorm and Dropout for better generalization
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.3),

    # Flatten to feed into Dense layers
    Flatten(),

    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    Dropout(0.4),
    BatchNormalization(),

    # Output layer (adjust number of units based on the number of classes)
    Dense(len(class_names), activation='softmax')
])

print("Model is successfully defined.")

In [ ]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print("Model is successfully compiled.")

In [ ]:
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)


reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

print("Function for early stopping and controlling learning rate is created.")

In [ ]:
# Train the model
history = model.fit(
    t_ds,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    callbacks=[early_stopping],
    verbose=1
)

print("Model is trained successfully.")

In [ ]:
# Evaluate the model
val_loss, val_accuracy = model.evaluate(test_ds)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")


print("Model is evaluated successfully.")

In [ ]:
# Plot training history
def plot_history(history):
    plt.figure(figsize=(12, 4))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='upper left')

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper left')

    plt.show()

# Plot the training history
plot_history(history)

In [ ]:
model.save("FP.h5")

print("Model is Saved successfully.")

In [ ]:
print("Let's test the model:")


import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model


model = load_model("FM.h5")  # Replace with your model path


# Load the image (resize as per your model's input)
img = load_img(r"D:\New Plant Diseases Dataset(Augmented)\valid\Potato___Late_blight\64fd9d16-28df-4dd7-9bb5-973bbd9ac6db___RS_LB 4654_180deg.JPG", target_size=(256, 256))

# Convert the image to a numpy array
img_array = img_to_array(img)

# Normalize the pixel values (if the model expects it)
img_array = img_array / 255.0  # Scale between 0 and 1

# Expand dimensions to match the input shape of the model (1, height, width, channels)
img_array = np.expand_dims(img_array, axis=0)

print(f"Processed Image Shape: {img_array.shape}")
# Predict the class probabilities
predictions = model.predict(img_array)

# Get the class with the highest probability
predicted_class = np.argmax(predictions, axis=1)

print(f"Predicted Class: {predicted_class[0]}")
print(f"Class Probabilities: {predictions}")


train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "Potato",
    shuffle = True,
    image_size = (256,256),
    batch_size= 32
)

class_names = train_ds.class_names
print(class_names)
print(f"Predicted Label: {class_names[predicted_class[0]]}")

import matplotlib.pyplot as plt

# Display an example image along with predicted and true labels
plt.imshow(img_array[0])
plt.title(f"Predicted: {class_names[predicted_class[0]]}")
plt.show()
